# T2.2 – Semantic Mapping

**Owner:** B — Julian Hardt  
**Task:** Map all attributes of the STATS19 collision table to domain ontology concepts and add the mappings to DBRepo via the REST API.

## Ontologies used

| Prefix | Ontology | URI | Justification |
|--------|----------|-----|--------------|
| `icity-te` | iCity Traffic Events Ontology | `https://w3id.org/icity/iCity-TE#` | The iCity Transportation Planning Suite of Ontologies (TPSO) is the most comprehensive and well-maintained OWL ontology for urban transportation, explicitly covering accident events, environmental conditions, road geometry, and severity classifications directly corresponding to STATS19 attributes. It is the most widely recommended domain-specific transport ontology in the academic literature (Katsumi & Fox, 2018). |
| `icity-otn` | iCity OpenTransportNet Ontology | `https://w3id.org/icity/OTN#` | Part of the iCity TPSO suite; covers road network concepts including road class, road type, junctions, speed limits, and administrative regions. Used where the Traffic Events ontology lacks a road-infrastructure concept. |
| `geo` | GeoSPARQL | `http://www.opengis.net/ont/geosparql#` | The OGC and W3C standard for geographic features; used for all spatial attributes (latitude, longitude, OSGB36 coordinates, and LSOA geometry). |
| `wgs84` | WGS84 Geo Positioning | `http://www.w3.org/2003/01/geo/wgs84_pos#` | W3C Basic Geo vocabulary; provides `lat` and `long` properties for WGS84 coordinates. |
| `time` | OWL-Time | `http://www.w3.org/2006/time#` | W3C recommendation for representing temporal concepts in Linked Data; used for date, time, and day-of-week attributes. |

## 0 – Configuration




In [9]:
# ── Fill these in ─────────────────────────────────────────────────────────────
DBREPO_ENDPOINT  = "https://test.dbrepo.tuwien.ac.at"
DBREPO_USERNAME  = "julian_345"
DBREPO_PASSWORD  =  "Juli_28121998"     # Settings → API Key on test.dbrepo.tuwien.ac.at

DATABASE_ID      = "82c19b39-246c-4409-b25c-8baf3a158a70"
TABLE_NAME       = "road_casualty_statistics_collision_last_5_years"
# ──────────────────────────────────────────────────────────────────────────────

## 1 – Install and import

In [3]:
%pip install dbrepo requests --quiet

Note: you may need to restart the kernel to use updated packages.


In [10]:
import requests
import json
from dbrepo.RestClient import RestClient

client = RestClient(
    endpoint=DBREPO_ENDPOINT,
    username=DBREPO_USERNAME,
    password=DBREPO_PASSWORD
)
print("Connected to DBRepo")

Connected to DBRepo


## 2 – Retrieve table and column IDs

In [11]:
# Get all tables in the database
tables = client.get_tables(database_id=DATABASE_ID)

# Find our table
target_table = None
for t in tables:
    if t.name == TABLE_NAME:
        target_table = t
        break

if target_table is None:
    raise ValueError(f"Table '{TABLE_NAME}' not found. Available tables: {[t.name for t in tables]}")

TABLE_ID = target_table.id
print(f"Found table: {TABLE_NAME}")
print(f"Table ID:    {TABLE_ID}")

Found table: road_casualty_statistics_collision_last_5_years
Table ID:    38dbe326-29a9-4503-a3d7-404b6b0ce31e


In [12]:
# Get full table details including columns
table_detail = client.get_table(database_id=DATABASE_ID, table_id=TABLE_ID)

# Build a dict: column_name -> column_id
column_ids = {col.name: col.id for col in table_detail.columns}

print(f"Found {len(column_ids)} columns:")
for name, cid in column_ids.items():
    print(f"  {name}: {cid}")

Found 44 columns:
  collision_index: b65a2297-035b-4d98-bc47-fb10ccddfc82
  collision_year: 88745318-9a61-401e-ad3c-a529cad36710
  collision_ref_no: 956f9859-bc2f-49d8-95b5-c1caf87fc3ce
  location_easting_osgr: f1e65d68-ae63-4ddf-bb33-8732e5721fba
  location_northing_osgr: 45148f53-183b-4cb6-b7fb-c60a2c537682
  longitude: 19a73401-feaf-45b1-8ee6-5217d5e602ed
  latitude: 085fa324-dd2d-440a-97ae-75751f7249c7
  police_force: 72fde083-9d55-463a-a894-7fdd18b2a961
  collision_severity: 63981ae5-1341-47cc-884c-b8d74612df3f
  number_of_vehicles: 998caf8c-a126-49f2-bd5a-a3e2c4c2ab54
  number_of_casualties: 33dc4dbc-b219-452a-8caf-d3daa0834e9a
  date: ee9c3486-576c-41cf-8071-ed640cf7d49c
  day_of_week: 8a5d3613-2c48-4c9a-9efc-d2e8b804241e
  time: 599fdace-164a-49bc-8eaa-7e4a918f4522
  local_authority_district: 596acbe2-8ae3-496a-be4c-a1053a4193cd
  local_authority_ons_district: ba5fa28c-3240-491d-a72f-d7c5b7494d8d
  local_authority_highway: a70cb954-d9ec-4592-aea7-a784b951b3ad
  local_authority_

## 3 – Define the ontology mappings

Each entry maps a column name to:
- `concept_uri`: the ontology concept URI
- `ontology`: short label for documentation
- `description`: human-readable explanation

In [13]:
ONTOLOGY_MAPPINGS = {
    # ── Identifiers & references ──────────────────────────────────────────────
    "collision_index": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#TrafficEvent",
        "ontology": "iCity-TE",
        "description": "Unique identifier for a recorded traffic accident event"
    },
    "collision_year": {
        "concept_uri": "http://www.w3.org/2006/time#year",
        "ontology": "OWL-Time",
        "description": "Calendar year in which the collision occurred"
    },
    "collision_ref_no": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#TrafficEvent",
        "ontology": "iCity-TE",
        "description": "Police reference number for the collision report"
    },
    # ── Spatial attributes ────────────────────────────────────────────────────
    "location_easting_osgr": {
        "concept_uri": "http://www.opengis.net/ont/geosparql#hasGeometry",
        "ontology": "GeoSPARQL",
        "description": "Easting coordinate in the Ordnance Survey National Grid (OSGB36) reference system"
    },
    "location_northing_osgr": {
        "concept_uri": "http://www.opengis.net/ont/geosparql#hasGeometry",
        "ontology": "GeoSPARQL",
        "description": "Northing coordinate in the Ordnance Survey National Grid (OSGB36) reference system"
    },
    "longitude": {
        "concept_uri": "http://www.w3.org/2003/01/geo/wgs84_pos#long",
        "ontology": "WGS84",
        "description": "WGS84 geographic longitude of the collision location"
    },
    "latitude": {
        "concept_uri": "http://www.w3.org/2003/01/geo/wgs84_pos#lat",
        "ontology": "WGS84",
        "description": "WGS84 geographic latitude of the collision location"
    },
    "lsoa_of_accident_location": {
        "concept_uri": "http://www.opengis.net/ont/geosparql#sfWithin",
        "ontology": "GeoSPARQL",
        "description": "Lower Super Output Area (LSOA) code identifying the census geography of the collision"
    },
    # ── Temporal attributes ───────────────────────────────────────────────────
    "date": {
        "concept_uri": "http://www.w3.org/2006/time#inXSDDate",
        "ontology": "OWL-Time",
        "description": "Calendar date on which the collision occurred (YYYY-MM-DD)"
    },
    "day_of_week": {
        "concept_uri": "http://www.w3.org/2006/time#dayOfWeek",
        "ontology": "OWL-Time",
        "description": "Day of the week on which the collision occurred (coded integer)"
    },
    "time": {
        "concept_uri": "http://www.w3.org/2006/time#inXSDTime",
        "ontology": "OWL-Time",
        "description": "Time of day at which the collision occurred (HH:MM)"
    },
    # ── Collision event attributes ────────────────────────────────────────────
    "collision_severity": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#AccidentSeverity",
        "ontology": "iCity-TE",
        "description": "Severity classification of the collision: Fatal (1), Serious (2), or Slight (3)"
    },
    "number_of_vehicles": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#numberOfVehiclesInvolved",
        "ontology": "iCity-TE",
        "description": "Count of vehicles involved in the collision"
    },
    "number_of_casualties": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#numberOfCasualties",
        "ontology": "iCity-TE",
        "description": "Total number of casualties (killed or injured) resulting from the collision"
    },
    "police_force": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#ReportingAgency",
        "ontology": "iCity-TE",
        "description": "Police force area responsible for recording and reporting the collision"
    },
    "did_police_officer_attend_scene_of_accident": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#EmergencyResponse",
        "ontology": "iCity-TE",
        "description": "Flag indicating whether a police officer attended the scene of the accident"
    },
    # ── Road network attributes ───────────────────────────────────────────────
    "road_type": {
        "concept_uri": "https://w3id.org/icity/OTN#RoadType",
        "ontology": "iCity-OTN",
        "description": "Type of road on which the collision occurred (e.g. single carriageway, dual carriageway, roundabout)"
    },
    "speed_limit": {
        "concept_uri": "https://w3id.org/icity/OTN#SpeedLimit",
        "ontology": "iCity-OTN",
        "description": "Posted speed limit in miles per hour at the collision location"
    },
    "first_road_class": {
        "concept_uri": "https://w3id.org/icity/OTN#RoadClassification",
        "ontology": "iCity-OTN",
        "description": "Classification of the first (primary) road at the collision site (e.g. Motorway, A road, B road)"
    },
    "first_road_number": {
        "concept_uri": "https://w3id.org/icity/OTN#Road",
        "ontology": "iCity-OTN",
        "description": "Numeric identifier of the first road at the collision site"
    },
    "second_road_class": {
        "concept_uri": "https://w3id.org/icity/OTN#RoadClassification",
        "ontology": "iCity-OTN",
        "description": "Classification of the second road at a junction collision site"
    },
    "second_road_number": {
        "concept_uri": "https://w3id.org/icity/OTN#Road",
        "ontology": "iCity-OTN",
        "description": "Numeric identifier of the second road at a junction collision site"
    },
    "trunk_road_flag": {
        "concept_uri": "https://w3id.org/icity/OTN#TrunkRoad",
        "ontology": "iCity-OTN",
        "description": "Flag indicating whether the road is classified as a trunk road"
    },
    "urban_or_rural_area": {
        "concept_uri": "https://w3id.org/icity/OTN#AreaType",
        "ontology": "iCity-OTN",
        "description": "Classification of the collision location as urban or rural"
    },
    # ── Junction attributes ───────────────────────────────────────────────────
    "junction_detail": {
        "concept_uri": "https://w3id.org/icity/OTN#Junction",
        "ontology": "iCity-OTN",
        "description": "Type of junction at or near the collision location"
    },
    "junction_detail_historic": {
        "concept_uri": "https://w3id.org/icity/OTN#Junction",
        "ontology": "iCity-OTN",
        "description": "Historic coding of junction type (legacy field from pre-2016 STATS19 reporting)"
    },
    "junction_control": {
        "concept_uri": "https://w3id.org/icity/OTN#TrafficControl",
        "ontology": "iCity-OTN",
        "description": "Type of traffic control present at the junction (e.g. traffic signals, give way, stop sign)"
    },
    # ── Environmental / site conditions ──────────────────────────────────────
    "light_conditions": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#LightCondition",
        "ontology": "iCity-TE",
        "description": "Ambient light conditions at the time of the collision (e.g. daylight, darkness with lighting)"
    },
    "weather_conditions": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#WeatherCondition",
        "ontology": "iCity-TE",
        "description": "Weather conditions at the time and location of the collision"
    },
    "road_surface_conditions": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#RoadSurfaceCondition",
        "ontology": "iCity-TE",
        "description": "Condition of the road surface at the time of the collision (e.g. dry, wet, frost)"
    },
    "special_conditions_at_site": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#RoadHazard",
        "ontology": "iCity-TE",
        "description": "Any special conditions or hazards present at the collision site"
    },
    "carriageway_hazards": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#RoadHazard",
        "ontology": "iCity-TE",
        "description": "Hazards present on the carriageway at the time of the collision"
    },
    "carriageway_hazards_historic": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#RoadHazard",
        "ontology": "iCity-TE",
        "description": "Historic coding of carriageway hazards (legacy field from pre-2016 STATS19 reporting)"
    },
    # ── Pedestrian crossing ───────────────────────────────────────────────────
    "pedestrian_crossing_human_control_historic": {
        "concept_uri": "https://w3id.org/icity/OTN#PedestrianCrossing",
        "ontology": "iCity-OTN",
        "description": "Historic coding of human-controlled pedestrian crossing presence (legacy STATS19 field)"
    },
    "pedestrian_crossing_physical_facilities_historic": {
        "concept_uri": "https://w3id.org/icity/OTN#PedestrianCrossing",
        "ontology": "iCity-OTN",
        "description": "Historic coding of physical pedestrian crossing facilities (legacy STATS19 field)"
    },
    "pedestrian_crossing": {
        "concept_uri": "https://w3id.org/icity/OTN#PedestrianCrossing",
        "ontology": "iCity-OTN",
        "description": "Type of pedestrian crossing facility present at the collision location"
    },
    # ── Administrative geography ──────────────────────────────────────────────
    "local_authority_district": {
        "concept_uri": "https://w3id.org/icity/OTN#AdministrativeRegion",
        "ontology": "iCity-OTN",
        "description": "Local authority district in which the collision occurred"
    },
    "local_authority_ons_district": {
        "concept_uri": "https://w3id.org/icity/OTN#AdministrativeRegion",
        "ontology": "iCity-OTN",
        "description": "ONS (Office for National Statistics) code for the local authority district"
    },
    "local_authority_highway": {
        "concept_uri": "https://w3id.org/icity/OTN#AdministrativeRegion",
        "ontology": "iCity-OTN",
        "description": "Highway authority responsible for the road where the collision occurred"
    },
    "local_authority_highway_current": {
        "concept_uri": "https://w3id.org/icity/OTN#AdministrativeRegion",
        "ontology": "iCity-OTN",
        "description": "Current highway authority code (updated from legacy coding)"
    },
    # ── Enhanced / adjusted severity fields ───────────────────────────────────
    "enhanced_severity_collision": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#AccidentSeverity",
        "ontology": "iCity-TE",
        "description": "Enhanced injury-based severity classification (where police forces have adopted injury-based reporting)"
    },
    "collision_injury_based": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#AccidentSeverity",
        "ontology": "iCity-TE",
        "description": "Flag indicating whether this record uses injury-based rather than outcome-based severity coding"
    },
    "collision_adjusted_severity_serious": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#AccidentSeverity",
        "ontology": "iCity-TE",
        "description": "Adjusted serious severity classification accounting for injury-based reporting adjustments"
    },
    "collision_adjusted_severity_slight": {
        "concept_uri": "https://w3id.org/icity/iCity-TE#AccidentSeverity",
        "ontology": "iCity-TE",
        "description": "Adjusted slight severity classification accounting for injury-based reporting adjustments"
    },
}

print(f"Defined mappings for {len(ONTOLOGY_MAPPINGS)} columns")

Defined mappings for 44 columns


## 4 – Check coverage

In [14]:
mapped   = set(ONTOLOGY_MAPPINGS.keys())
in_db    = set(column_ids.keys())

missing  = in_db - mapped    # columns in DB but no mapping defined
extra    = mapped - in_db    # mappings defined but column not in DB

print(f"Columns in DB:        {len(in_db)}")
print(f"Mappings defined:     {len(mapped)}")
print(f"Missing mappings:     {sorted(missing)}")
print(f"Extra (not in DB):    {sorted(extra)}")

Columns in DB:        44
Mappings defined:     44
Missing mappings:     []
Extra (not in DB):    []


In [16]:
# Diagnostic: check what methods exist on client and what concepts are available
print("=== Client methods with 'concept' or 'ontology' ===")
print([m for m in dir(client) if 'concept' in m.lower() or 'ontolog' in m.lower() or 'column' in m.lower()])

print("\n=== Available ontologies in DBRepo ===")
try:
    ontologies = client.get_ontologies()
    for o in ontologies:
        print(f"  {o}")
except Exception as e:
    print(f"  Error: {e}")

print("\n=== Available concepts (first 10) ===")
try:
    concepts = client.get_concepts()
    for c in concepts[:10]:
        print(f"  {c}")
except Exception as e:
    print(f"  Error: {e}")

=== Client methods with 'concept' or 'ontology' ===
['get_concepts', 'get_ontologies', 'update_table_column']

=== Available ontologies in DBRepo ===
  Error: Failed to get ontologies: response code: 404 is not 200 (OK): {"type":"about:blank","title":"Not Found","status":404,"detail":"No static resource api/v1/ontology.","instance":"/api/v1/ontology","properties":null}

=== Available concepts (first 10) ===
  Error: Failed to get concepts: response code: 404 is not 200 (OK): {"type":"about:blank","title":"Not Found","status":404,"detail":"No static resource api/v1/concept.","instance":"/api/v1/concept","properties":null}


## 5 – Push mappings to DBRepo via REST API

DBRepo stores semantic concepts at the column level via `update_column_concept`.

In [17]:
results = []

for col_name, mapping in ONTOLOGY_MAPPINGS.items():
    if col_name not in column_ids:
        print(f"  SKIP  {col_name} (not found in DB)")
        continue

    col_id = column_ids[col_name]
    concept_uri = mapping["concept_uri"]

    try:
        client.update_table_column(
            database_id=DATABASE_ID,
            table_id=TABLE_ID,
            column_id=col_id,
            concept_uri=concept_uri
        )
        print(f"  OK    {col_name} → {concept_uri}")
        results.append({"column": col_name, "status": "ok", "concept_uri": concept_uri})

    except Exception as e:
        print(f"  FAIL  {col_name} — {e}")
        results.append({"column": col_name, "status": f"error: {e}", "concept_uri": concept_uri})

print(f"\nDone. {sum(1 for r in results if r['status'] == 'ok')}/{len(results)} mappings applied.")

  OK    collision_index → https://w3id.org/icity/iCity-TE#TrafficEvent
  OK    collision_year → http://www.w3.org/2006/time#year
  OK    collision_ref_no → https://w3id.org/icity/iCity-TE#TrafficEvent
  OK    location_easting_osgr → http://www.opengis.net/ont/geosparql#hasGeometry
  OK    location_northing_osgr → http://www.opengis.net/ont/geosparql#hasGeometry
  OK    longitude → http://www.w3.org/2003/01/geo/wgs84_pos#long
  OK    latitude → http://www.w3.org/2003/01/geo/wgs84_pos#lat
  OK    lsoa_of_accident_location → http://www.opengis.net/ont/geosparql#sfWithin
  OK    date → http://www.w3.org/2006/time#inXSDDate
  OK    day_of_week → http://www.w3.org/2006/time#dayOfWeek
  OK    time → http://www.w3.org/2006/time#inXSDTime
  OK    collision_severity → https://w3id.org/icity/iCity-TE#AccidentSeverity
  OK    number_of_vehicles → https://w3id.org/icity/iCity-TE#numberOfVehiclesInvolved
  OK    number_of_casualties → https://w3id.org/icity/iCity-TE#numberOfCasualties
  OK    police

## 6 – Verify: read back the mappings

In [18]:
# Re-fetch the table and confirm concepts are stored
table_updated = client.get_table(database_id=DATABASE_ID, table_id=TABLE_ID)

print(f"{'Column':<50} {'Concept URI'}")
print("-" * 100)
for col in table_updated.columns:
    concept = getattr(col, 'concept_uri', None) or getattr(col, 'concept', None) or '—'
    print(f"{col.name:<50} {concept}")

Column                                             Concept URI
----------------------------------------------------------------------------------------------------
collision_index                                    https://w3id.org/icity/iCity-TE#TrafficEvent
collision_year                                     http://www.w3.org/2006/time#year
collision_ref_no                                   https://w3id.org/icity/iCity-TE#TrafficEvent
location_easting_osgr                              http://www.opengis.net/ont/geosparql#hasGeometry
location_northing_osgr                             http://www.opengis.net/ont/geosparql#hasGeometry
longitude                                          http://www.w3.org/2003/01/geo/wgs84_pos#long
latitude                                           http://www.w3.org/2003/01/geo/wgs84_pos#lat
police_force                                       https://w3id.org/icity/iCity-TE#ReportingAgency
collision_severity                                 https://w3id.org/ic

## 7 – Export mapping table for README / report

In [20]:
%pip install tabulate --quiet

import pandas as pd

df = pd.DataFrame([
    {
        "Column": col,
        "Ontology": info["ontology"],
        "Concept URI": info["concept_uri"],
        "Description": info["description"]
    }
    for col, info in ONTOLOGY_MAPPINGS.items()
])

print(df.to_markdown(index=False))

df.to_csv("semantic_mapping_t2_2.csv", index=False)
print("\nSaved to semantic_mapping_t2_2.csv")

Note: you may need to restart the kernel to use updated packages.
| Column                                           | Ontology   | Concept URI                                              | Description                                                                                             |
|:-------------------------------------------------|:-----------|:---------------------------------------------------------|:--------------------------------------------------------------------------------------------------------|
| collision_index                                  | iCity-TE   | https://w3id.org/icity/iCity-TE#TrafficEvent             | Unique identifier for a recorded traffic accident event                                                 |
| collision_year                                   | OWL-Time   | http://www.w3.org/2006/time#year                         | Calendar year in which the collision occurred                                                           |
| collisio